In [1]:
pip install --upgrade \ git+ssh://git@labdev.colibri-quantum.com/colibritd-rd/landscape_characterization.git

  Cloning ssh://****@labdev.colibri-quantum.com/colibritd-rd/landscape_characterization.git to /tmp/pip-req-build-rpxahcqp
  Running command git clone --filter=blob:none --quiet 'ssh://****@labdev.colibri-quantum.com/colibritd-rd/landscape_characterization.git' /tmp/pip-req-build-rpxahcqp
  Resolved ssh://****@labdev.colibri-quantum.com/colibritd-rd/landscape_characterization.git to commit 72263220398acf4076ab66101190029be05d42a6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.8 MB/s eta 0:00:00
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached gudhi-3.12.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_

# Imports

In [2]:
from landscape_characterization import barren_plateaus as bp
from landscape_characterization import landscape_visualization as lv
from landscape_characterization import exploratory_landscape_analysis as ela

# Barren plateaus analysis

## Important functions

### Cost function

In [ ]:
def cost_function_exp(
    use_state_vector: bool,
    current_params: npt.NDArray[np.float64],
    isa_circuit,
    isa_obs_list: Pauli,
    run_mode: str,
    shots: int,
    backend,
):
    angles = current_params[1:]

    # Single Pauli observable
    pauli_string = str(isa_obs_list)
    # Single Walsh index
    pauli_monomial_index = pauli_to_index_zstring(pauli_string)

    if use_state_vector is False:

        estimator = get_estimator(run_mode, shots)
        pub = (isa_circuit, isa_obs_list, angles)
        result = estimator.run(pubs=[pub]).result()
        expectation_values = result[0].data.evs

    else:

        first_pauli_op = pauli_string.lstrip("I")[0]
        num_qubits = isa_circuit.num_qubits

        if len(pauli_string) != num_qubits:
            raise ValueError(
                f"Observable has {len(pauli_string)} qubits "
                f"but circuit has {num_qubits}"
            )

        if run_mode == "ideal":

            qc_wht = isa_circuit.assign_parameters(angles)
            rotate_for_paulis(
                qc_wht,
                [first_pauli_op] * num_qubits,
            )

            qc_wht.save_statevector()
            backend = AerSimulator(method="statevector")
            psi = backend.run(qc_wht).result().get_statevector()
            proba_density = np.abs(psi) ** 2
            expectation_value_all_pauli_monomials = (
                fast_walsh_transform(proba_density)
            )
            expectation_values = (
                expectation_value_all_pauli_monomials[
                    pauli_monomial_index
                ]
            )
        else:
            qc_wht = isa_circuit.copy()

            rotate_for_paulis(
                qc_wht,
                [first_pauli_op] * num_qubits,
            )

            qc_wht = qc_wht.assign_parameters(angles)
            qc_meas = qc_wht.copy()
            qc_meas.measure_all()
            job = backend.run(qc_meas, shots=shots)
            counts = job.result().get_counts()

            proba_dens_hat = counts_to_prob_vector(
                counts,
                num_qubits,
            )
            expectation_values = (
                expectations_for_indices_from_proba_density(
                    proba_dens_hat,
                    [pauli_monomial_index],
                )[0]
            )

    return expectation_values

### Transpilation

In [ ]:
def transpile_exp(
    circuit,
    isa_obs_list,
    run_mode="ideal",
    backend: Optional[AerSimulator] = None,
    optimization_level: int = 3,
    layout_method: str = "trivial",
):

    # No transpilation
    if run_mode.lower() in ("ideal", "shot", "none", "classical"):
        return circuit, isa_obs_list

    # Backend
    if backend is None:
        backend = AerSimulator()

    # Noise backend
    if run_mode.lower() in ("fake", "gatenoise"):
        noise_model = NoiseModel.from_backend(FakeAlgiers())
        backend = AerSimulator(noise_model=noise_model)

    # Pass manager
    preset_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
        backend=backend,
        layout_method=layout_method,
    )

    # Transpile circuit
    isa_circuit = preset_manager.run(circuit)

    # Apply layout to Pauli observable
    layout = isa_circuit.layout
    isa_obs_trans = isa_obs_list.apply_layout(layout=layout)

    return isa_circuit, isa_obs_trans

### Quantum circuit generation

In [ ]:
def generate_circuits_exp(
    n_qubits: int,
    depth: int,
    ansatz_function: Callable,
    **ansatz_options: Dict,
) -> QuantumCircuit:

    circuit = ansatz_function(
        "F_0",
        n_qubits,
        depth,
        **ansatz_options,
    )
    return circuit

### Parameters generation

In [ ]:
def generate_params_exp(n_qubits, depth):

    # n_params = n_qubits * depth + 1
    # params = np.random.uniform(0, 2 * np.pi, n_params)
    
    ## For Floquet ansatz:
    hn1 = generate_h_list(
        n_qubits=n_qubits, use_mode="3", phase="mbl", seed= np.random.randint(1000000000)
    )  # if use_mode != "3" or phase != "mbl" it needs to be changed in floquet_ansatz_generalized() as well (in Ansatze.py)
    params = np.insert(hn1, 0, 1)

    return params

def sample_once():
    theta = generate_params(n_qubits, depth)
    return theta

## Cost function builder

In [ ]:
def my_cost_builder(context, **kwargs):

    circuits = context["circuits"]
    observables = context["observables"]

    isa_circuit, isa_obs = transpile_exp(
        circuits,
        observables,
        run_mode=kwargs["run_mode"],
    )

    def cost_only_params(current_params):
        return cost_function_exp(
            use_state_vector=kwargs["use_state_vector"],
            current_params=current_params,
            isa_circuit=isa_circuit,
            isa_obs_list=isa_obs,
            run_mode=kwargs["run_mode"],
            shots=kwargs["shots"],
            backend=kwargs["backend"],
        )

    return cost_only_params

## Run analysis

In [ ]:
results = bp.barren_plateaus_analysis(
    analysis_type="layerwise_qubits_padding",
    N_qubits=[2,4,6,8,12],
    N_layers=[3,5,7,10],
    initial_Pauli_string= Pauli('IY'),
    padding_types=["identity", "linear_half"],
    cost_function_builder=my_cost_builder,
    generate_params=generate_params_exp,
    generate_circuits=generate_circuits_exp,
    use_state_vector=True,
    run_mode=run_mode,
    shots=None,
    Ansatz=floquet_mbl_ansatz,
    backend=backend,
)